# 03 — Query optimization: what actually made it fast

The claim this notebook defends is the one in the README: **interactive percentiles over
36M rows, served from free-tier hosting.** Two paths, measured separately.

| Path | What the app is doing | What it reads |
|---|---|---|
| **Serving** | drawing a peer group's distribution | `peer_stats`, 3.3 MB, in the repo |
| **Drill-down** | placing one provider inside that distribution | fact Parquet, remote, over HTTP |

**The metric is bytes read, not seconds.** The fact tables are published as a GitHub
Release asset and read with `httpfs`, so DuckDB fetches them with HTTP range requests. On
that path, sort order and projection pushdown decide how much data crosses the network —
which is a cost that a warm local benchmark hides completely. So the levers below are
measured in bytes, on a real HTTP server, with the bytes counted as they are served.

Everything here rests on [`docs/schema.md`](../docs/schema.md) for the model and
[`docs/build_report.md`](../docs/build_report.md) for what the build measured.

In [1]:
import contextlib
import time
from pathlib import Path

import duckdb
import pandas as pd

from cms_outliers.measure import drop_page_cache, serve_directory

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
CORE = REPO / "data" / "parquet"
SERVING = REPO / "data" / "serving"
RAW = REPO / "data" / "raw"

PART_B = CORE / "fact_part_b_service" / "year=2023" / "data.parquet"
PART_B_CSV = RAW / "part_b_2023_full.csv"

pd.set_option("display.width", 120)
pd.set_option("display.max_colwidth", 60)

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs")
threads = con.execute("SELECT current_setting('threads')").fetchone()[0]
print(f"DuckDB {duckdb.__version__}, {threads} threads")
print(f"fact_part_b_service: {PART_B.stat().st_size:,} bytes")
print(f"part_b CSV:          {PART_B_CSV.stat().st_size:,} bytes")

DuckDB 1.5.5, 15 threads
fact_part_b_service: 319,531,174 bytes
part_b CSV:          3,062,332,720 bytes


In [2]:
def timed(sql, params=(), runs=3):
    """Median of `runs` warm executions, after one throwaway run to warm the cache.

    Warm on purpose: every timing in this notebook that is not explicitly labelled cold is
    a warm timing, and saying so once here is better than qualifying each number.
    """
    con.execute(sql, params).fetchall()
    elapsed = []
    for _ in range(runs):
        start = time.perf_counter()
        con.execute(sql, params).fetchall()
        elapsed.append(time.perf_counter() - start)
    return sorted(elapsed)[len(elapsed) // 2]


# The peer group used throughout, picked from the data rather than hardcoded.
PEER = con.execute(
    f"""SELECT specialty, hcpcs_code, place_of_service
        FROM read_parquet('{PART_B.as_posix()}')
        GROUP BY ALL ORDER BY count(*) DESC LIMIT 1"""
).fetchone()
print("peer group under test:", PEER)

peer group under test: ('Nurse Practitioner', '99214', 'O')


## Lever 1 — CSV to typed Parquet

The starting point. Same aggregate, same engine, two storage formats.

ADR 0001's original consequences list led with this lever as the likely biggest win. The
exploration in notebook 02 already suggested otherwise, and this is where that gets
settled.

In [3]:
QUERY = """
SELECT count(*) AS n_peers,
       quantile_cont({col}, 0.90) AS p90
FROM {src}
WHERE {spec} = ? AND {code} = ? AND {pos} = ?
"""

csv_sql = QUERY.format(
    src=f"read_csv('{PART_B_CSV.as_posix()}')",
    col="Avg_Mdcr_Stdzd_Amt", spec="Rndrng_Prvdr_Type", code="HCPCS_Cd", pos="Place_Of_Srvc",
)
parquet_sql = QUERY.format(
    src=f"read_parquet('{PART_B.as_posix()}')",
    col="avg_medicare_standardized", spec="specialty", code="hcpcs_code", pos="place_of_service",
)

assert con.execute(csv_sql, PEER).fetchone() == con.execute(parquet_sql, PEER).fetchone()

lever1 = pd.DataFrame([
    {"format": "CSV",     "bytes on disk": PART_B_CSV.stat().st_size, "seconds": timed(csv_sql, PEER)},
    {"format": "Parquet", "bytes on disk": PART_B.stat().st_size,     "seconds": timed(parquet_sql, PEER)},
])
lever1["x smaller"] = (lever1["bytes on disk"].iloc[0] / lever1["bytes on disk"]).round(1)
lever1["x faster"] = (lever1["seconds"].iloc[0] / lever1["seconds"]).round(1)
lever1

,format,bytes on disk,seconds,x smaller,x faster
0,CSV,3062332720,0.444083,1.0,1.0
1,Parquet,319531174,0.003470,9.6,128.0


### Cold cache

Every timing above is warm. That matters, because a large part of the CSV cost is reading
3 GB off disk, and a warm page cache hides it — the honest comparison needs both.

Dropping the page cache on macOS needs `sudo purge`, so the cell below reports whether it
actually managed to, rather than relabelling a warm number as a cold one.

In [4]:
if drop_page_cache():
    cold_csv = timed(csv_sql, PEER, runs=1)
    if drop_page_cache():
        cold_parquet = timed(parquet_sql, PEER, runs=1)
        print(f"cold CSV     {cold_csv:.3f}s")
        print(f"cold Parquet {cold_parquet:.3f}s  ({cold_csv / cold_parquet:.0f}x)")
else:
    print("Page cache NOT dropped — `sudo purge` needs a password, so no cold timing was")
    print("taken. Run `sudo purge` yourself and re-execute this cell to fill it in.")
    print()
    print("This is the open cold-cache question in ADR 0001. It matters less than it did:")
    print("the drill-down path below is measured in bytes over HTTP, which is not a")
    print("cache-warmth question at all.")

Page cache NOT dropped — `sudo purge` needs a password, so no cold timing was
taken. Run `sudo purge` yourself and re-execute this cell to fill it in.

This is the open cold-cache question in ADR 0001. It matters less than it did:
the drill-down path below is measured in bytes over HTTP, which is not a
cache-warmth question at all.


## The drill-down path, over HTTP

From here on the fact table is read the way the deployed app will read it — over HTTP, with
range requests, from a server that counts every byte it serves. The network is local so the
numbers are about file layout and nothing else.

`traffic.reset()` is called immediately before each measured query, because DuckDB fetches
the Parquet footer once when it first opens a file and that is setup, not query cost.

In [5]:
stack = contextlib.ExitStack()
CORE_URL, traffic = stack.enter_context(serve_directory(CORE))
REMOTE_B = f"{CORE_URL}/fact_part_b_service/year=2023/data.parquet"

con.execute(f"CREATE OR REPLACE VIEW remote_part_b AS SELECT * FROM read_parquet('{REMOTE_B}')")
con.execute("SELECT count(*) FROM remote_part_b").fetchall()  # open the file, read the footer
print(f"footer and metadata: {traffic.summary()}")

footer and metadata: 524,288 bytes in 2 requests


## Lever 2 — sort order and zone-map pruning

The fact table is sorted by `(specialty, hcpcs_code, place_of_service)` — the peer key. Each
Parquet row group stores the min and max of every column, so a reader can skip whole row
groups whose range cannot contain the value being filtered on. That only works if matching
rows are *contiguous*: in a randomly ordered file every row group's range spans nearly the
whole domain, and nothing can be skipped.

ADR 0001's first amendment predicted a single-peer-group filter should touch ~1 row group of
79. This is the test of that prediction, against an unsorted copy of the identical data.

Watch the file sizes in the next cell too: they are not the same, and the reason is the same
reason the pruning works.

In [6]:
UNSORTED = REPO / "data" / "parquet-unsorted"
unsorted_path = UNSORTED / "fact_part_b_service" / "year=2023" / "data.parquet"
unsorted_path.parent.mkdir(parents=True, exist_ok=True)

# Same rows, same columns, same compression, same row-group size — only the order differs.
con.execute(f"""
    COPY (SELECT * FROM read_parquet('{PART_B.as_posix()}') ORDER BY random())
    TO '{unsorted_path.as_posix()}' (FORMAT parquet, COMPRESSION zstd)
""")
print(f"sorted   {PART_B.stat().st_size:,} bytes")
print(f"unsorted {unsorted_path.stat().st_size:,} bytes")

sorted   319,531,174 bytes
unsorted 407,343,217 bytes


In [7]:
def row_groups_matching_specialty(path, specialty):
    """Row groups whose `specialty` zone map cannot rule the value out."""
    return con.execute(
        f"""SELECT count(DISTINCT row_group_id) AS candidates,
                   (SELECT count(DISTINCT row_group_id)
                    FROM parquet_metadata('{path}')) AS total
            FROM parquet_metadata('{path}')
            WHERE path_in_schema = 'specialty'
              AND stats_min <= ? AND stats_max >= ?""",
        [specialty, specialty],
    ).fetchone()

for label, path in [("sorted", PART_B.as_posix()), ("unsorted", unsorted_path.as_posix())]:
    candidates, total = row_groups_matching_specialty(path, PEER[0])
    print(f"{label:9} {candidates:>3} of {total} row groups can contain {PEER[0]!r}")

sorted      8 of 79 row groups can contain 'Nurse Practitioner'
unsorted   79 of 79 row groups can contain 'Nurse Practitioner'


In [8]:
UNSORTED_URL_STACK = contextlib.ExitStack()
UNSORTED_URL, unsorted_traffic = UNSORTED_URL_STACK.enter_context(serve_directory(UNSORTED))
remote_unsorted = f"{UNSORTED_URL}/fact_part_b_service/year=2023/data.parquet"
con.execute(
    f"CREATE OR REPLACE VIEW remote_unsorted AS SELECT * FROM read_parquet('{remote_unsorted}')"
)
con.execute("SELECT count(*) FROM remote_unsorted").fetchall()

drilldown = """
SELECT npi, tot_srvcs, tot_srvcs_pct
FROM {src}
WHERE specialty = ? AND hcpcs_code = ? AND place_of_service = ?
ORDER BY tot_srvcs DESC LIMIT 10
"""

rows = []
for label, view, tr in [
    ("sorted", "remote_part_b", traffic),
    ("unsorted", "remote_unsorted", unsorted_traffic),
]:
    sql = drilldown.format(src=view)
    con.execute(sql, PEER).fetchall()
    tr.reset()
    con.execute(sql, PEER).fetchall()
    rows.append({"layout": label, "bytes read": tr.total_bytes, "requests": tr.n_requests})

lever2 = pd.DataFrame(rows)
lever2["% of file"] = (100 * lever2["bytes read"] / PART_B.stat().st_size).round(2)
lever2

,layout,bytes read,requests,% of file
0,sorted,1773802,207,0.56
1,unsorted,46729759,822,14.62


## Lever 4 — projection pushdown

Parquet stores each column separately, so a reader can fetch only the columns a query names.
The app's drill-down needs three of the thirteen fact columns. The comparison is the same
filter, `SELECT *` against the named projection — on the sorted file, so pruning is held
constant and only the projection changes.

The per-column footprint in [`build_report.md`](../docs/build_report.md) is the ceiling on
this: a query can only avoid reading what a column actually costs.

In [9]:
rows = []
for label, projection in [("SELECT *", "*"), ("3 columns", "npi, tot_srvcs, tot_srvcs_pct")]:
    sql = f"""SELECT {projection} FROM remote_part_b
              WHERE specialty = ? AND hcpcs_code = ? AND place_of_service = ?"""
    con.execute(sql, PEER).fetchall()
    traffic.reset()
    con.execute(sql, PEER).fetchall()
    rows.append({"projection": label, "bytes read": traffic.total_bytes,
                 "requests": traffic.n_requests})

lever4 = pd.DataFrame(rows)
lever4["x less"] = (lever4["bytes read"].iloc[0] / lever4["bytes read"]).round(1)
lever4

,projection,bytes read,requests,x less
0,SELECT *,4940672,203,1.0
1,3 columns,1365926,205,3.6


## Lever 6 — the serving path

The app's first question on every interaction is *what does this peer group look like?* Two
ways to answer it: compute the percentiles over the fact table now, or read the row that was
computed at build time.

This is the lever the whole storage decision was made for (ADR 0001, point 3: a
pre-aggregated table is the right fix for a percentile scan, and an index is not).

Three paths, not two, because the comparison turns on which one is the honest baseline:

1. the live percentile over the **local** fact file,
2. the live percentile over the **remote** fact file — what a deployed app would actually do
   if it had no serving layer,
3. the `peer_stats` lookup.

In [10]:
live_sql = f"""
SELECT count(*) AS n_peers,
       quantile_cont(tot_srvcs, 0.50) AS p50,
       quantile_cont(tot_srvcs, 0.90) AS p90,
       quantile_cont(tot_srvcs, 0.99) AS p99
FROM read_parquet('{PART_B.as_posix()}')
WHERE specialty = ? AND hcpcs_code = ? AND place_of_service = ?
"""

peer_stats_path = (SERVING / "peer_stats" / "year=2023" / "data.parquet").as_posix()
served_sql = f"""
SELECT n_rows AS n_peers, p50, p90, p99
FROM read_parquet('{peer_stats_path}')
WHERE dataset = 'part_b' AND measure = 'tot_srvcs'
  AND specialty = ? AND code = ? AND place_of_service = ?
"""

live_remote_sql = live_sql.replace(f"read_parquet('{PART_B.as_posix()}')", "remote_part_b")

live = con.execute(live_sql, PEER).fetchone()
served = con.execute(served_sql, PEER).fetchone()
assert live == con.execute(live_remote_sql, PEER).fetchone()
assert live[0] == served[0], "the paths disagree on the peer group size"
print("all three paths agree:", served)

# Bytes over HTTP for the live remote path, against peer_stats' whole file size.
con.execute(live_remote_sql, PEER).fetchall()
traffic.reset()
con.execute(live_remote_sql, PEER).fetchall()
live_remote_bytes = traffic.total_bytes

lever6 = pd.DataFrame([
    {"path": "live percentile, local facts", "bytes read": PART_B.stat().st_size,
     "seconds": timed(live_sql, PEER)},
    {"path": "live percentile, remote facts", "bytes read": live_remote_bytes,
     "seconds": timed(live_remote_sql, PEER)},
    {"path": "peer_stats lookup", "bytes read": Path(peer_stats_path).stat().st_size,
     "seconds": timed(served_sql, PEER)},
])
lever6["x faster than remote"] = (lever6["seconds"].iloc[1] / lever6["seconds"]).round(1)
lever6

all three paths agree: (82339, 80.0, 326.0, 819.0)


,path,bytes read,seconds,x faster than remote
0,"live percentile, local facts",319531174,0.004516,15.2
1,"live percentile, remote facts",555329,0.068836,1.0
2,peer_stats lookup,3465122,0.002896,23.8


## Lever 3 — partition pruning: deliberately not measured

The Parquet is Hive-partitioned by year, which is the right layout: it is how CMS publishes
the data, it costs nothing, and a year filter would skip whole directories. But only 2023 is
loaded, so there is exactly one partition and nothing to skip.

Rather than stage a demonstration on a second copy of the same year, this is left unmeasured
and the ADR says so. A partitioning speedup measured against one partition would be a number
about the benchmark, not about the data.

## What this adds up to

Both paths, end to end, with the numbers from above.

In [11]:
summary = pd.DataFrame([
    {"lever": "1. CSV → typed Parquet",
     "metric": f"{lever1['x faster'].iloc[1]:.0f}x faster, "
               f"{lever1['x smaller'].iloc[1]:.0f}x smaller on disk"},
    {"lever": "2. Sort order → zone-map pruning",
     "metric": f"{lever2['bytes read'].iloc[1] / lever2['bytes read'].iloc[0]:.0f}x fewer "
               f"bytes over HTTP than the same data unsorted"},
    {"lever": "4. Projection pushdown",
     "metric": f"{lever4['x less'].iloc[1]:.1f}x fewer bytes than SELECT *"},
    {"lever": "6. Precomputed peer_stats",
     "metric": f"{lever6['x faster than remote'].iloc[2]:.1f}x faster than the same "
               f"percentile computed over the remote facts"},
    {"lever": "3. Partition pruning", "metric": "not measured — one partition loaded"},
])
summary

,lever,metric
0,1. CSV → typed Parquet,"128x faster, 10x smaller on disk"
1,2. Sort order → zone-map pruning,26x fewer bytes over HTTP than the same data unsorted
2,4. Projection pushdown,3.6x fewer bytes than SELECT *
3,6. Precomputed peer_stats,23.8x faster than the same percentile computed over the ...
4,3. Partition pruning,not measured — one partition loaded


In [12]:
stack.close()
UNSORTED_URL_STACK.close()
print("servers stopped")

servers stopped
